In [1]:
import pandas as pd
import ast
import os

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
for genus_name in keep_genus:
    print(genus_name)
    replicon_info = pd.read_csv(f'/active-data/analysis_results/chr_pla/genus/IMG_PR_plasmid/{genus_name}/IMGPR_plasmid_data.tsv', sep='\t')
    folder = f'/active-data/analysis_results/chr_pla/genus/IMG_PR_plasmid/{genus_name}'
    fasta_path = os.path.join(folder, "IMGPR_metagenome_plasmids.fasta")
    if not os.path.exists(fasta_path) or os.path.getsize(fasta_path) == 0:
        continue
        
    data_dir = f'/active-data/analysis_results/chr_pla/genus/IMG_PR_plasmid/{genus_name}/annotations'
    os.makedirs(data_dir, exist_ok=True)
    
    os.chdir(folder)
    %time os.system(f'amrfinder -n IMGPR_metagenome_plasmids.fasta -o {data_dir}/amrfinder_IMGPR_metagenome_plasmid_result.txt --threads 16 --quiet')
    
    amr_resu = pd.read_csv(f'{data_dir}/amrfinder_IMGPR_metagenome_plasmid_result.txt', sep='\t')
    amr_resu = amr_resu[amr_resu['Type'] == 'AMR']
    amr_resu['plasmid_id'] = amr_resu['Contig id'].str.split('|').str[0]
    
    amr_resu.rename(columns={'Type': 'AMR_type'}, inplace=True)
    replicon_type = pd.merge(replicon_info[['plasmid_id']], amr_resu[['plasmid_id', 'AMR_type']], on='plasmid_id', how='left').fillna('non-AMR')
    replicon_type.to_csv(f'{data_dir}/IMGPR_metagenome_plasmid_AMRtyper_results.tsv', sep = '\t',)

Escherichia
CPU times: user 1.09 ms, sys: 1.01 ms, total: 2.1 ms
Wall time: 13 s
Klebsiella
CPU times: user 3.4 ms, sys: 25 μs, total: 3.43 ms
Wall time: 32.2 s
Staphylococcus
CPU times: user 14.1 ms, sys: 1.25 ms, total: 15.4 ms
Wall time: 1min 33s
Pseudomonas
CPU times: user 1.64 ms, sys: 1.04 ms, total: 2.67 ms
Wall time: 15.9 s
Bacillus
CPU times: user 1.08 ms, sys: 1.06 ms, total: 2.14 ms
Wall time: 10.1 s
Salmonella
CPU times: user 137 μs, sys: 1.92 ms, total: 2.06 ms
Wall time: 7.93 s
Streptococcus
CPU times: user 2.31 ms, sys: 67 μs, total: 2.38 ms
Wall time: 12 s
Streptomyces
CPU times: user 798 μs, sys: 995 μs, total: 1.79 ms
Wall time: 1.91 s
Acinetobacter
CPU times: user 951 μs, sys: 1 ms, total: 1.95 ms
Wall time: 5.25 s
Enterococcus
CPU times: user 1.19 ms, sys: 1.02 ms, total: 2.21 ms
Wall time: 5.81 s
Bordetella
Enterobacter
CPU times: user 793 μs, sys: 0 ns, total: 793 μs
Wall time: 2.71 s
Xanthomonas
CPU times: user 667 μs, sys: 994 μs, total: 1.66 ms
Wall time: 1.1 s